# Introduction to LangChain

## Initial setup

### Set API key for Groq
Click [here](https://console.groq.com/keys) to create API key for Groq, if not already created.

In [1]:
import os, json, re, getpass
from dotenv import load_dotenv

load_dotenv( override=True)

True

In [2]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [4]:
# if "TEST_API_KEY" not in os.environ:
#     os.environ["TEST_API_KEY"] = getpass.getpass("TEST API Key: ")

In [3]:
if os.environ["GROQ_API_KEY"]:
    print(f"Groq API Key exists and begins {os.environ["GROQ_API_KEY"][:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


## LangChain Components

### LLM / ChatModel

In [ ]:
!ollama pull llama3.2 ## TO UPDATE THE MODEL

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


**Note** on **init_chat_model**: init_chat_model is just a helper method and under the hood, it will still be calling the specific Chat models only (like ChatOpenAI etc.). The only benefit of using init_chat_model is that the initialization is standard across providers, which is useful. 

See the source code of this method here for better details. Note that line 79 has init_chat_model() function and if model and model_provider are specified (which we do in class), then it returns an instance of _ConfigurableModel class (line 332). Then this class definition (line 554), returns the model in line 610 using _init_chat_model_helper() method. This method definition (line 339) actually returns ChatOllama() model instance (line 400). So it's the same thing.


In [ ]:
#Using LangChain
from langchain.chat_models import init_chat_model

model_name = "llama-3.1-8b-instant"
llm = init_chat_model(model_name, model_provider="groq")

In [7]:
llm_response = llm.invoke("what is ECG?")
llm_response

AIMessage(content="ECG stands for Electrocardiogram. It is a medical test that measures the electrical activity of the heart. An ECG is a non-invasive test that records the electrical impulses that control the heartbeat. It is often used to diagnose and monitor heart conditions, such as:\n\n1. **Arrhythmias**: Abnormal heart rhythms\n2. **Myocardial infarction** (heart attack)\n3. **Cardiac hypertrophy**: Thickening of the heart muscle\n4. **Cardiac conduction disorders**: Abnormalities in the electrical conduction system of the heart\n5. **Electrolyte imbalances**: Abnormal levels of electrolytes such as potassium, sodium, or calcium\n\nDuring an ECG, small electrodes are placed on the skin to detect the electrical signals produced by the heart. The signals are then recorded on a graph, which can be used to diagnose and monitor heart conditions.\n\nAn ECG typically measures:\n\n1. **Heart rate**: The number of times the heart beats per minute\n2. **P-QRS-T waves**: The different stage

In [8]:
print("type of response", type(llm_response))

type of response <class 'langchain_core.messages.ai.AIMessage'>


In [9]:
# print(llm_response.content)
# display(llm_response.response_metadata)
display(llm_response.usage_metadata)

{'input_tokens': 40, 'output_tokens': 393, 'total_tokens': 433}

In [10]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
                  Ensure proper sentence structure, clarity, and readability.\
                  Retain the core message of the original text while making the necessary corrections"),
    HumanMessage(content="hey can you send me that report by tomorrow thx"),
]

ai_response = llm.invoke(messages)
print(ai_response.content)

Here's the corrected version:

"Hey, can you send me that report by tomorrow, please?"

I made the following corrections:

- Added a comma after "tomorrow" to improve sentence structure and clarity.
- Changed "thx" to "please" for proper spelling and etiquette. 

This revised sentence is more polite and clear in its request.


In [12]:
ai_response

AIMessage(content='Here\'s the corrected version:\n\n"Hey, can you send me that report by tomorrow, please?"\n\nI made the following corrections:\n\n- Added a comma after "tomorrow" to improve sentence structure and clarity.\n- Changed "thx" to "please" for proper spelling and etiquette. \n\nThis revised sentence is more polite and clear in its request.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 89, 'total_tokens': 161, 'completion_time': 0.111260809, 'prompt_time': 0.009088769, 'queue_time': 0.046340771, 'total_time': 0.120349578}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_46fc01befd', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--770d7f04-5a29-4670-a491-1324eb8d6510-0', usage_metadata={'input_tokens': 89, 'output_tokens': 72, 'total_tokens': 161})

In [13]:
# followup conversation
messages.append(ai_response)

In [14]:
#Ask a follow-up question
messages.append(HumanMessage(content="can you make the tone a bit informal"))

In [15]:
ai_response = llm.invoke(messages)
# print(ai_response)
print(ai_response.content)

Here's the corrected version with a more informal tone:

"Hey, can you send me that report by tomorrow, no worries?"

I made the following corrections:

- Exchanged "please" for "no worries" to give the tone a more relaxed and friendly feel.
- No other major corrections were needed.

This revised sentence still conveys the same message as the original but with a more casual tone.


#### Doing without LangChain

In [16]:
#Without LangChain - how would we initialize our LLM?
from openai import OpenAI

model_name = "llama-3.1-8b-instant"
llm_api = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")

In [17]:
# #Below is LangChain's messages format
# messages = [

#     SystemMessage(content="Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
#                   Ensure proper sentence structure, clarity, and readability.\
#                   Retain the core message of the original text while making the necessary corrections"),
#     HumanMessage(content="hey can you send me that report by tomorrow thx"),
# ]

#Below is messages in OpenAI format
messages_openai = [
    {'role':"system", 'content':"Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
     Ensure proper sentence structure, clarity, and readability.\
     Retain the core message of the original text while making the necessary corrections"},
     
     {'role':"user", 'content':"hey can you send me that report by tomorrow thx"}
]


In [19]:
ai_response_openai = llm_api.chat.completions.create(model= model_name,
                                messages=messages_openai)

In [20]:
ai_response_openai

ChatCompletion(id='chatcmpl-cc7dbb63-91f8-461a-8d5b-9da97087b0f1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here\'s the corrected text:\n\n"Hey, can you send me that report by tomorrow, thanks?"', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1757828511, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_46fc01befd', usage=CompletionUsage(completion_tokens=21, prompt_tokens=89, total_tokens=110, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.04566775, prompt_time=0.00891238, completion_time=0.024773629, total_time=0.033686009), usage_breakdown=None, x_groq={'id': 'req_01k53bwxw6eg38yzj9bvme8ksv'})

In [21]:
ai_response_openai_formatted = ai_response_openai.choices[0].message.content
print(ai_response_openai_formatted)

Here's the corrected text:

"Hey, can you send me that report by tomorrow, thanks?"


In [22]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'}]

In [23]:
#Append the AI message
messages_openai.append(
    {'role': "assistant",
     'content': ai_response_openai_formatted}
)

In [24]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'},
 {'role': 'assistant',
  'content': 'Here\'s the corrected text:\n\n"Hey, can you send me that report by tomorrow, thanks?"'}]

In [25]:
#Ask a follow-up question
#LangChain version below
# messages.append(HumanMessage(content="can you make the tone a bit informal"))

#OpenAI version below
messages_openai.append(
    {'role':"user",
     'content':"can you make the tone a bit informal"}
)

In [26]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'},
 {'role': 'assistant',
  'content': 'Here\'s the corrected text:\n\n"Hey, can you send me that report by tomorrow, thanks?"'},
 {'role': 'user', 'content': 'can you make the tone a bit informal'}]

In [28]:
ai_response_openai = llm_api.chat.completions.create(
    model=model_name,
    messages=messages_openai
)

In [29]:
type(ai_response_openai)

openai.types.chat.chat_completion.ChatCompletion

In [30]:
print(ai_response_openai.choices[0].message.content)

Here's the text with a slightly more informal tone:

"Hey, can you send me that report by tomorrow? Thanks!"


In [28]:
#Explain concept of context length here - Context length = input tokens + completion tokens 

### Output Parsers

In [32]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

parser = StrOutputParser()

*StrOutputParser* is a runnable object.

In [33]:
result = llm.invoke(messages)

parser.invoke(result)

'Here\'s the revised version with a slightly more informal tone:\n\n"Hey, can you send me that report by tomorrow, thanks?"\n\nI kept the main corrections from the previous version and made the following adjustments to make the tone slightly more informal:\n\n- Changed "please" to "thanks" for a more casual tone.'

In [ ]:
result.content ## NOT DOING THIS ANYMORE, USING THE PARSER INSTEAD

'Here\'s the revised version with a slightly more informal tone:\n\n"Hey, can you send me that report by tomorrow, thanks?"\n\nI kept the main corrections from the previous version and made the following adjustments to make the tone slightly more informal:\n\n- Changed "please" to "thanks" for a more casual tone.'

In [ ]:
result

AIMessage(content='Here\'s the revised text:\n\n"Hey, can you send me that report by tomorrow? Thanks!"\n\nChanges made:\n- Changed "Hi" back to "Hey" (more informal greeting)\n- Removed the comma after "tomorrow" (less formal and more conversational)\n- Changed "please" to "Thanks!" (more casual and friendly expression)\n\nThis revised text has a friendly, yet still professional tone.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 192, 'total_tokens': 276, 'completion_time': 0.111757862, 'prompt_time': 0.529207673, 'queue_time': 0.047274416, 'total_time': 0.640965535}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_46fc01befd', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--ebedec27-c41c-4dca-8b11-32627688e1ec-0', usage_metadata={'input_tokens': 192, 'output_tokens': 84, 'total_tokens': 276})

In [44]:
messages = [
    SystemMessage(content="""You are an expert in writing analysis. You will receive a message from a user, and your job is to evaluate the text based on the following attributes:
1. clarity: Is the message clear, or unclear?
2. grammar_quality: Are there any grammatical issues? Possible values: correct, minor issues, major issues.
3. tone: Analyze whether the tone is neutral, formal, or informal.
4. suggestions: Offer brief improvement suggestions for clarity, grammar, or tone.

Return a structured JSON object with these four attributes. Wrap the JSON between ```json tags"""),
    HumanMessage(content="Hey, could you please send me that report by tomorrow? Thank you.")
]
response = llm.invoke(messages)

In [45]:
response

AIMessage(content='```json\n{\n  "clarity": "unclear",\n  "grammar_quality": "minor issues",\n  "tone": "informal",\n  "suggestions": [\n    "Consider being more specific about what report you need.",\n    "You may want to use a more formal closing phrase, such as \'I appreciate your assistance\' instead of \'Thank you\'.",\n    "If possible, include a reminder of your deadline, such as \'Please send the report by tomorrow at [time]\'."\n  ]\n}\n```', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 166, 'total_tokens': 274, 'completion_time': 0.269580011, 'prompt_time': 0.01344544, 'queue_time': 0.05017999, 'total_time': 0.283025451}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_46fc01befd', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--804b6ef1-2e59-4d91-9be0-5d4dec4b5c1d-0', usage_metadata={'input_tokens': 166, 'output_tokens': 108, 'total_tokens': 274})

In [46]:
print(response.content)

```json
{
  "clarity": "unclear",
  "grammar_quality": "minor issues",
  "tone": "informal",
  "suggestions": [
    "Consider being more specific about what report you need.",
    "You may want to use a more formal closing phrase, such as 'I appreciate your assistance' instead of 'Thank you'.",
    "If possible, include a reminder of your deadline, such as 'Please send the report by tomorrow at [time]'."
  ]
}
```


In [47]:
json_response = JsonOutputParser().invoke(response.content)
json_response

{'clarity': 'unclear',
 'grammar_quality': 'minor issues',
 'tone': 'informal',
 'suggestions': ['Consider being more specific about what report you need.',
  "You may want to use a more formal closing phrase, such as 'I appreciate your assistance' instead of 'Thank you'.",
  "If possible, include a reminder of your deadline, such as 'Please send the report by tomorrow at [time]'."]}

In [51]:
var1 = '{"clarity": "unclear"}'
print(var1)

{"clarity": "unclear"}


In [52]:
var1

'{"clarity": "unclear"}'

In [46]:
# JsonOutputParser().invoke(var1)

In [50]:
type(json_response)

dict

In [48]:
type(var1)

str

In [53]:
print(response.content)

```json
{
  "clarity": "unclear",
  "grammar_quality": "minor issues",
  "tone": "informal",
  "suggestions": [
    "Consider being more specific about what report you need.",
    "You may want to use a more formal closing phrase, such as 'I appreciate your assistance' instead of 'Thank you'.",
    "If possible, include a reminder of your deadline, such as 'Please send the report by tomorrow at [time]'."
  ]
}
```


In [54]:
json_response['clarity']

'unclear'

### Chain (LCEL)

In [55]:
chain = llm | JsonOutputParser()

chain.invoke(messages)

{'clarity': 'unclear',
 'grammar_quality': 'minor issues',
 'tone': 'informal',
 'suggestions': ["Consider including more context about what report you're referring to.",
  'If you have a specific request or deadline, it would be helpful to include that information.',
  "You could rephrase 'Thank you' to something more specific, like 'I appreciate your help with this'."]}

### PromptTemplate

In [52]:
# my_str = "Hello world, this is the tone - {tone}"

In [53]:
# my_str.format(tone = "Happy")

In [54]:
# tone = "Happy"
# my_str = f"Hello world, this is the tone - {tone}"
# print(my_str)

In [56]:
my_str = """Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: {tone}

Communication Style: {communication_style}

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style."""

In [57]:
print(my_str.format(tone="Happy", communication_style = "Formal"))

Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: Happy

Communication Style: Formal

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style.


In [58]:
from langchain_core.prompts import ChatPromptTemplate
system_message_template = """Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: {tone}

Communication Style: {communication_style}

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style."""

#Defining a ChatPromptTemplate
template = ChatPromptTemplate([
    ("system", system_message_template),
    ("human", "{user_input}"),
])

In [60]:
template.input_variables

['communication_style', 'tone', 'user_input']

In [59]:
#How does this work without LCEL = LangChain Expression LangChain

In [61]:
tone = 'Rewrite the message in a professional, polite, and structured manner. \
Suitable for business emails, official reports, or any context requiring formality and respect.'

communication_style = 'Messages should be clear, structured, and formal or neutral depending on the context. \
Introductions, conclusions, and appropriate sign-offs should be added if missing.'

user_input = """Can u send me the data by eod pls?"""

In [62]:
#First step
formatted_template = template.invoke({"communication_style": communication_style,
    "tone":tone,
    "user_input":user_input
})
formatted_template

ChatPromptValue(messages=[SystemMessage(content='Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\nEnsure proper sentence structure, clarity, and readability.\n\nTone Adjustment: Rewrite the message in a professional, polite, and structured manner. Suitable for business emails, official reports, or any context requiring formality and respect.\n\nCommunication Style: Messages should be clear, structured, and formal or neutral depending on the context. Introductions, conclusions, and appropriate sign-offs should be added if missing.\n\nRetain the core message of the original text while making the necessary corrections and tone adjustments.\nSuggest appropriate phrasing and formatting based on the tone and communication style.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Can u send me the data by eod pls?', additional_kwargs={}, response_metadata={})])

In [63]:
type(formatted_template)

langchain_core.prompt_values.ChatPromptValue

In [64]:
#Second step
response = llm.invoke(formatted_template)

In [65]:
type(response)

langchain_core.messages.ai.AIMessage

In [66]:
response.content

"Here's a rewritten version of the message in a professional, polite, and structured manner:\n\nDear [Recipient's Name],\n\nI would greatly appreciate it if you could provide the necessary data by the end of the day (EOD) today. This will enable me to proceed with the required tasks on a timely basis.\n\nThank you for your prompt attention to this matter.\n\nBest regards,\n[Your Name]\n\nAlternatively, you could also use a more direct but still polite approach:\n\nDear [Recipient's Name],\n\nCould you please provide the data by the end of the day (EOD) today? This will help me to meet the project deadlines.\n\nThank you for your cooperation.\n\nBest regards,\n[Your Name]\n\nIf you want to make the message even more concise, you could use:\n\nDear [Recipient's Name],\n\nPlease provide the data by the end of the day (EOD) today to enable timely progress on the project.\n\nThank you.\n\nBest regards,\n[Your Name]"

In [67]:
#Third step
final_parsed_result = StrOutputParser().invoke(response)
final_parsed_result

"Here's a rewritten version of the message in a professional, polite, and structured manner:\n\nDear [Recipient's Name],\n\nI would greatly appreciate it if you could provide the necessary data by the end of the day (EOD) today. This will enable me to proceed with the required tasks on a timely basis.\n\nThank you for your prompt attention to this matter.\n\nBest regards,\n[Your Name]\n\nAlternatively, you could also use a more direct but still polite approach:\n\nDear [Recipient's Name],\n\nCould you please provide the data by the end of the day (EOD) today? This will help me to meet the project deadlines.\n\nThank you for your cooperation.\n\nBest regards,\n[Your Name]\n\nIf you want to make the message even more concise, you could use:\n\nDear [Recipient's Name],\n\nPlease provide the data by the end of the day (EOD) today to enable timely progress on the project.\n\nThank you.\n\nBest regards,\n[Your Name]"

In [67]:
# #With LCEL
# proof_read_chain = template | llm ##RunnableSequence
# final_response = proof_read_chain.invoke(
#     {"communication_style": communication_style,
#     "tone":tone,
#     "user_input":user_input
# })
# final_response
# type(final_response)

In [69]:
#With LCEL
proof_read_chain = template | llm | StrOutputParser()
proof_read_chain

ChatPromptTemplate(input_variables=['communication_style', 'tone', 'user_input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['communication_style', 'tone'], input_types={}, partial_variables={}, template='Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\nEnsure proper sentence structure, clarity, and readability.\n\nTone Adjustment: {tone}\n\nCommunication Style: {communication_style}\n\nRetain the core message of the original text while making the necessary corrections and tone adjustments.\nSuggest appropriate phrasing and formatting based on the tone and communication style.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_input'], input_types={}, partial_variables={}, template='{user_input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x127299790>, async_client=<groq.

In [70]:
type(proof_read_chain)

langchain_core.runnables.base.RunnableSequence

In [71]:
final_response = proof_read_chain.invoke(
    {"communication_style": communication_style,
    "tone":tone,
    "user_input":user_input
})
final_response

"Here's a rewritten version of the message in a professional and polite tone:\n\nDear [Recipient],\n\nCould you please provide me with the requested data by the end of the day (EOD) today? I would greatly appreciate it if you could arrange for it to be sent over as soon as possible.\n\nThank you for your prompt attention to this matter.\n\nBest regards,\n[Your Name]\n\nAlternatively, if you need to communicate this request in a more concise manner, you can use:\n\nDear [Recipient],\n\nI kindly request that you send the data by the end of the day today. Your prompt assistance in this matter is greatly appreciated.\n\nThank you,\n[Your Name]\n\nPlease note that you may need to adjust the tone and style according to your specific requirements and relationship with the recipient."

In [71]:
# proof_read_chain.input_schema.model_json_schema()

In [73]:
tone_map = {
    "Formal": "Rewrite the message in a professional, polite, and structured manner. Suitable for business emails, official reports, or any context requiring formality and respect.",
    "Informal": "Rewrite in a casual, friendly, and conversational style. Appropriate for personal communications, friendly chats, or informal emails.",
    "Neutral": "Rewrite in a balanced tone that is neither overly formal nor too casual. Suitable for most general communications where a middle-ground tone is required."
}
communication_style_map = {
    "Email": "Messages should be clear, structured, and formal or neutral depending on the context. Introductions, conclusions, and appropriate sign-offs should be added if missing.",
    "General": "This covers most forms of communication and will aim for clarity and coherence. The tone can vary as per the user's choice.",
    "Instant Messaging": "Focus on brevity, clarity, and informality, using conversational phrasing suitable for quick back-and-forth exchanges.",
    "Business Instant Messaging": "Maintain a professional but conversational tone. Messages should be concise and efficient, avoiding unnecessary formalities but keeping the language respectful."
}

In [74]:
tone = "Formal" # Formal, Informal, Neutral
communication_style = "Email" # Email, General, Instant Messaging, Business Instant Messaging
user_input = """Can u send me the data by eod pls?"""

chain_output = proof_read_chain.invoke(dict(
    tone=tone_map[tone],
    communication_style = communication_style_map[communication_style],
    user_input = user_input
))
print(chain_output)

Here's a rewritten version of the message in a professional and polite tone:

Dear [Recipient's Name],

I would appreciate it if you could provide me with the required data by the end of the day today. Please let me know if there are any issues or concerns that may prevent timely delivery.

Thank you for your prompt attention to this matter.

Best regards,
[Your Name]

Alternatively, if you'd like to convey the request in a more concise manner, you could use:

Dear [Recipient's Name],

Could you please provide the required data by the end of the day today?

Thank you for your assistance.

Best regards,
[Your Name]

This revised message maintains a professional tone while conveying the original request in a clear and structured manner.
